# Mapping the Language Family Tree

How big are the world's language families, and how deep does German's lineage go?
This notebook explores Glottolog's phylogeny via `db.families` and walks the
ancestor chain for individual languages.

**Requirements**
```
pip install "languages-of-the-world[examples]"
```

In [ ]:
%pip install -q pandas plotly matplotlib


In [ ]:
import pandas as pd
import plotly.express as px

import low

## 1 · Load the database

In [ ]:
db = low.LanguagesOfTheWorld()
print(db)
print(f"Family tree nodes: {len(db.families)}")

## 2 · Top-level root families

`db.families.roots()` returns families with no parent — the trunks of the tree.

In [ ]:
roots = db.families.roots()
print(f"Root families: {len(roots)}")
for fam in sorted(roots, key=lambda f: f.label)[:8]:
    print(f"  {fam.label} ({fam.glottocode}) — {len(fam.children)} direct subgroups")

## 3 · Count descendant languages per root family

In [ ]:
def count_descendant_languages(family, seen=None):
    """Recursively count all languages under a family node."""
    seen = seen or set()
    if family.glottocode in seen:
        return 0
    seen.add(family.glottocode)
    total = len(family.languages)
    for child in family.children:
        total += count_descendant_languages(child, seen)
    return total

root_rows = [
    {
        "family": fam.label,
        "glottocode": fam.glottocode,
        "subgroups": len(fam.children),
        "languages": count_descendant_languages(fam),
    }
    for fam in roots
]

root_df = pd.DataFrame(root_rows).sort_values("languages", ascending=False)
root_df.head(15)

## 4 · Treemap of top families by language count

In [ ]:
top_roots = root_df.head(20)

fig = px.treemap(
    top_roots,
    path=["family"],
    values="languages",
    title="Top 20 Root Language Families by Descendant Language Count",
    color="languages",
    color_continuous_scale="Blues",
)
fig.update_layout(margin=dict(l=10, r=10, t=40, b=10))
fig.show()

## 5 · Walk German's lineage

Starting from `Language.family`, follow `.parent` links up to the root.
`depth` and `ancestors` provide the same information as properties.

In [ ]:
deu = db.languages.get("deu")
print(f"{deu.label} — glottocode: {deu.glottocode}")
print(f"Immediate family: {deu.family.label}")
print()
print("Lineage (leaf → root):")
node = deu.family
while node:
    indent = "  " * node.depth
    print(f"{indent}{node.label} (depth={node.depth})")
    node = node.parent

print()
print("Via .ancestors property:")
for anc in deu.family.ancestors:
    print(f"  {anc.label}")
print(f"Root family: {deu.family.root.label}")

## 6 · Lookup by glottocode or label

In [ ]:
indo = db.families.get("indo1319")
print(f"get('indo1319') → {indo.label}")
print(f"Direct child languages: {len(indo.languages)}")
print(f"Direct subgroups: {len(indo.children)}")
for child in indo.children[:5]:
    print(f"  {child.label} ({len(child.languages)} langs, {len(child.children)} subgroups)")

## 7 · Families with the most immediate member languages

In [ ]:
immediate = sorted(
    (f for f in db.families if f.languages),
    key=lambda f: len(f.languages),
    reverse=True,
)[:15]

imm_df = pd.DataFrame([
    {"family": f.label, "glottocode": f.glottocode, "immediate_languages": len(f.languages)}
    for f in immediate
])

fig2 = px.bar(
    imm_df,
    x="immediate_languages",
    y="family",
    orientation="h",
    title="Families with Most Direct Member Languages",
    labels={"immediate_languages": "Languages", "family": ""},
)
fig2.update_layout(yaxis={"categoryorder": "total ascending"}, height=500)
fig2.show()

## 8 · Summary

In [ ]:
kin = db.languages.get("kin")
print(f"{kin.label}: family={kin.family.label if kin.family else '—'}, endangerment={kin.endangerment}")
print(f"Total languages with a family assignment: {sum(1 for l in db.languages if l.family):,}")